In [ ]:
!pip install datasets>=2.6.1
!pip install transformers
!pip install evaluate>=0.30
!pip install opencc
!pip install -q git+https://github.com/huggingface/peft.git
!pip install -U torchao
!pip install jiwer
!pip install -q optuna

#!pip install -U torchao peft transformers accelerate
#!pip install -q soundfile

# 卸載所有相關舊版本
#!pip uninstall -y torch torchvision torchaudio torchcodec
#!pip install librosa
# 安裝最新穩定版 PyTorch
#!pip install torch torchvision torchaudio --upgrade

# 安裝 torchcodec 最新版本
#!pip install torchcodec --upgrade
#!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 24.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.5 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.8 MB/s eta 0:00:00


In [ ]:
# LoRA Fine-tune

from peft import LoraConfig, get_peft_model
from datasets import load_from_disk
from transformers import WhisperProcessor
import evaluate
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import WhisperForConditionalGeneration
from transformers import Seq2SeqTrainer
import re
import unicodedata
from transformers import EarlyStoppingCallback # 1. 新增引入 Early Stopping
import opencc


import random
import numpy as np

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


# 1. 載入基礎模型
# 這裡不需要 load_in_8bit，建議 device_map="auto" 讓它自動分配 GPU
model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-tiny",
    device_map="auto"
)

processor = WhisperProcessor.from_pretrained("openai/whisper-tiny", language="zh", task="transcribe")

train_ds = load_from_disk("/content/drive/MyDrive/ds/augmented_ds")
test_ds = load_from_disk("/content/drive/MyDrive/ds/acupointTCM")

model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="zh", task="transcribe")
model.generation_config.suppress_tokens = [] # 防止生成時出現無聲片段的 bug

metric = evaluate.load("cer")

_CC = opencc.OpenCC('s2t')   # Simplified → Traditional (Taiwan phrases)

# ── 全形轉半形 ─────────────────────────────────────────────────────────────────

def _full2half(text: str) -> str:
    """
    全形 ASCII (U+FF01–U+FF5E) → 半形 (U+0021–U+007E)
    全形空格 (U+3000) → 半形空格
    """
    result = []
    for ch in text:
        cp = ord(ch)
        if cp == 0x3000:           # 　→ 空格
            result.append(' ')
        elif 0xFF01 <= cp <= 0xFF5E:  # ！…～、全形英數標點
            result.append(chr(cp - 0xFEE0))
        else:
            result.append(ch)
    return ''.join(result)

# ── 數字轉中文 ─────────────────────────────────────────────────────────────────

_ZH_DIGITS = '零一二三四五六七八九'

def _section_zh(n: int) -> str:
    """將 0–9,999 轉為中文（內部使用）。"""
    parts, need_zero = [], False
    for char, w in [('千', 1000), ('百', 100), ('十', 10), ('', 1)]:
        d, n = divmod(n, w)
        if d:
            if need_zero:
                parts.append('零')
            parts.append(_ZH_DIGITS[d] + char)
            need_zero = False
        elif parts:
            need_zero = True
    return ''.join(parts)

def _int_zh(n: int) -> str:
    """將非負整數轉為中文數字。"""
    if n == 0:
        return '零'
    if n < 10_000:
        return _section_zh(n)
    if n < 100_000_000:
        wan, rest = divmod(n, 10_000)
        if rest == 0:
            return _section_zh(wan) + '萬'
        if rest < 1_000:
            return _section_zh(wan) + '萬零' + _section_zh(rest)
        return _section_zh(wan) + '萬' + _section_zh(rest)
    yi, rest = divmod(n, 100_000_000)
    if rest == 0:
        return _int_zh(yi) + '億'
    if rest < 10_000_000:
        return _int_zh(yi) + '億零' + _int_zh(rest)
    return _int_zh(yi) + '億' + _int_zh(rest)

def _num_to_zh(m: re.Match) -> str:
    s = m.group()
    if '.' in s:
        left, right = s.split('.', 1)
        int_zh = _int_zh(int(left)) if left else '零'
        dec_zh = ''.join(_ZH_DIGITS[int(c)] for c in right)
        return int_zh + '點' + dec_zh
    return _int_zh(int(s))

# ── 標點正則 ───────────────────────────────────────────────────────────────────

# 全形轉半形後，，！？；：已變為半形，因此兩者都納入
_PUNCT_RE = re.compile(
    r'[。、…—～·「」『』【】《》〈〉'
    r'.,!?;"\'`@#$%^&*+=<>~_|/()\[\]{}\-\\]'
)
_TRAILING_RE = re.compile(
    r'[。、…—～·「」『』【】《》〈〉'
    r'.,!?;"\'`@#$%^&*+=<>~_|/()\[\]{}\-\\\s]+'
    r'$'
)

# ── 主函式 ─────────────────────────────────────────────────────────────────────

def normalize_text(text: str) -> str:
    # 規則 0a：簡體 → 繁體（臺灣用詞）
    text = _CC.convert(text)

    # 規則 0b：全形 → 半形（含英數、標點、空格）
    #          ：(U+FF1A) 在此一併轉為 :，後續無需特別處理全形冒號
    text = _full2half(text)

    # 規則 5：'兩' → '2'（接著在規則 3 統一轉為「二」）
    text = text.replace('兩', '2')

    # 規則 4+3：先匹配小數再匹配整數，統一轉中文
    text = re.sub(r'\d+\.\d+', _num_to_zh, text)
    text = re.sub(r'\d+', _num_to_zh, text)

    # 規則 2：英文轉大寫
    text = text.upper()

    # 規則 1b：句尾標點移除
    text = _TRAILING_RE.sub('', text)

    # 規則 1c：其餘標點轉空格（冒號 ':' 不在此 pattern，得以保留）
    text = _PUNCT_RE.sub(' ', text)

    # 整理多餘空白
    text = re.sub(r'\s+', '', text).strip()

    return text

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    pred_str = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]

    cer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"cer": cer}

from transformers import Seq2SeqTrainingArguments

# 為了節省顯存並穩定訓練，手動將模型設為訓練模式並開啟 Gradient Checkpointing
model.train()

# 2. 設定 LoRA 參數
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
    lora_dropout=0,
    bias="none",
)

# 3. 將 LoRA 套用到模型
model = get_peft_model(model, config)
model.print_trainable_parameters()

_train_size = len(train_ds["train"])
_steps_per_epoch = max(1, _train_size // 16)

training_args = Seq2SeqTrainingArguments(
    seed=42,  # 固定 Trainer 內部的隨機種子
    data_seed=42,  # 固定資料 shuffle 的隨機種子
    output_dir="/content/drive/MyDrive/model/tcmWhisper7",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=6e-4,
    num_train_epochs = 10,
    gradient_checkpointing=False,
    fp16=True,
    weight_decay = 0.00,
    lr_scheduler_type = "linear",
    optim = "adamw_torch",

    # --- 評估與存檔策略 ---
    eval_strategy="steps",
    save_strategy="steps",      # 配合 load_best_model_at_end
    save_steps=_steps_per_epoch,
    eval_steps=_steps_per_epoch,
    logging_steps=_steps_per_epoch,
    warmup_steps=_steps_per_epoch,

    # --- 你原本的參數 (已補回) ---
    per_device_eval_batch_size=16, # 驗證時的 Batch Size
    predict_with_generate=False,   # 讓模型在驗證時實際生成文字 (計算 CER 必備)
    #generation_max_length=300,    # 生成長度限制

    # --- 推薦的優化參數 ---
    save_total_limit=2,           # 只留最新的 2 個存檔，避免塞爆硬碟
    load_best_model_at_end=True,  # 訓練結束載入 CER 最低的
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to=["tensorboard"],
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_ds["train"],
    eval_dataset=test_ds["validation"],
    data_collator=data_collator,
    #compute_metrics=compute_metrics,

    # --- 關鍵修正：修復 Tokenizer 遺失問題 ---
    # 這裡一定要傳入 processor (包含了 feature_extractor 和 tokenizer)
    # 這樣上傳到 Hugging Face 時才會包含 vocab.json 等必要檔案
    processing_class=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

model.generation_config.use_cache = False

trainer.train()

#trainer.train(resume_from_checkpoint=True)

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

trainable params: 1,081,344 || all params: 38,841,984 || trainable%: 2.7840


Step,Training Loss,Validation Loss
294,0.668436,0.491767
588,0.201393,0.469403
882,0.102978,0.408889
1176,0.059407,0.397586
1470,0.034401,0.400084
1764,0.021297,0.424783
2058,0.013300,0.434160


TrainOutput(global_step=2058, training_loss=0.15731605299476054, metrics={'train_runtime': 8078.1932, 'train_samples_per_second': 5.823, 'train_steps_per_second': 0.364, 'total_flos': 8.6192400236544e+17, 'train_loss': 0.15731605299476054, 'epoch': 7.0})

In [ ]:
# ============================================================
# Whisper LoRA Fine-tune + Optuna 超參數搜尋 (Colab 版本)
# ============================================================

# 在 Colab cell 中執行前，請先安裝 optuna：

import os
import gc
import json
import random
import re
import unicodedata
from dataclasses import dataclass
from typing import Any, Dict, List, Union

import numpy as np
import torch
import opencc
import evaluate
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from peft import LoraConfig, get_peft_model
from datasets import load_from_disk
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
    EarlyStoppingCallback,
)

# ------------------------------------------------------------
# 0. Colab / Google Drive 掛載（非 Colab 環境會自動略過，不會報錯）
# ------------------------------------------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("非 Colab 環境，略過 Google Drive 掛載")

# ------------------------------------------------------------
# 1. 固定隨機種子
# ------------------------------------------------------------
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ------------------------------------------------------------
# 2. 載入 processor / 資料集（整個搜尋過程只需載入一次）
# ------------------------------------------------------------
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny", language="zh", task="transcribe")

#train_ds = load_from_disk("/content/drive/MyDrive/ds/augmented_ds")
train_ds = load_from_disk("/content/drive/MyDrive/ds/optuna_trainds")
test_ds = load_from_disk("/content/drive/MyDrive/ds/acupointTCM")

# ------------------------------------------------------------
# 3. 載入基礎模型（整個搜尋過程只需載入一次）。
#    LoRA 不會改動 base model 的權重（凍結），所以每個 trial
#    結束後只要 unload() 把 adapter 拆掉，就能拿回乾淨的 base
#    model 重複使用，不必每個 trial 都重新從硬碟讀檔 50 次。
# ------------------------------------------------------------
base_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-tiny",
    device_map="auto",
)
base_model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="zh", task="transcribe")
base_model.generation_config.suppress_tokens = []
base_model.generation_config.use_cache = False

# ------------------------------------------------------------
# 4. 文字正規化 / CER 計算（給最終訓練階段用；搜尋階段為了加快
#    速度，只用 eval_loss 當搜尋目標，不做生成式評估）
# ------------------------------------------------------------
metric = evaluate.load("cer")
_CC = opencc.OpenCC('s2t')

def _full2half(text: str) -> str:
    result = []
    for ch in text:
        cp = ord(ch)
        if cp == 0x3000:
            result.append(' ')
        elif 0xFF01 <= cp <= 0xFF5E:
            result.append(chr(cp - 0xFEE0))
        else:
            result.append(ch)
    return ''.join(result)

_ZH_DIGITS = '零一二三四五六七八九'

def _section_zh(n: int) -> str:
    parts, need_zero = [], False
    for char, w in [('千', 1000), ('百', 100), ('十', 10), ('', 1)]:
        d, n = divmod(n, w)
        if d:
            if need_zero:
                parts.append('零')
            parts.append(_ZH_DIGITS[d] + char)
            need_zero = False
        elif parts:
            need_zero = True
    return ''.join(parts)

def _int_zh(n: int) -> str:
    if n == 0:
        return '零'
    if n < 10_000:
        return _section_zh(n)
    if n < 100_000_000:
        wan, rest = divmod(n, 10_000)
        if rest == 0:
            return _section_zh(wan) + '萬'
        if rest < 1_000:
            return _section_zh(wan) + '萬零' + _section_zh(rest)
        return _section_zh(wan) + '萬' + _section_zh(rest)
    yi, rest = divmod(n, 100_000_000)
    if rest == 0:
        return _int_zh(yi) + '億'
    if rest < 10_000_000:
        return _int_zh(yi) + '億零' + _int_zh(rest)
    return _int_zh(yi) + '億' + _int_zh(rest)

def _num_to_zh(m: re.Match) -> str:
    s = m.group()
    if '.' in s:
        left, right = s.split('.', 1)
        int_zh = _int_zh(int(left)) if left else '零'
        dec_zh = ''.join(_ZH_DIGITS[int(c)] for c in right)
        return int_zh + '點' + dec_zh
    return _int_zh(int(s))

_PUNCT_RE = re.compile(
    r'[。、…—～·「」『』【】《》〈〉'
    r'.,!?;"\'`@#$%^&*+=<>~_|/()\[\]{}\-\\]'
)
_TRAILING_RE = re.compile(
    r'[。、…—～·「」『』【】《》〈〉'
    r'.,!?;"\'`@#$%^&*+=<>~_|/()\[\]{}\-\\\s]+'
    r'$'
)

def normalize_text(text: str) -> str:
    text = _CC.convert(text)
    text = _full2half(text)
    text = text.replace('兩', '2')
    text = re.sub(r'\d+\.\d+', _num_to_zh, text)
    text = re.sub(r'\d+', _num_to_zh, text)
    text = text.upper()
    text = _TRAILING_RE.sub('', text)
    text = _PUNCT_RE.sub(' ', text)
    text = re.sub(r'\s+', '', text).strip()
    return text

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    pred_str = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]

    cer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"cer": cer}

# ------------------------------------------------------------
# 5. 搜尋階段共用設定
# ------------------------------------------------------------
SEARCH_BATCH_SIZE = 16
SEARCH_EPOCHS = 3
N_TRIALS = 50
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"]

OUTPUT_ROOT = "/content/drive/MyDrive/optuna/optuna_model"
STUDY_DB_PATH = "/content/drive/MyDrive/optuna/optuna_whisper_lora.db"
BEST_PARAMS_PATH = "/content/drive/MyDrive/optuna/optuna_whisper_lora_best_params.json"

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(os.path.dirname(STUDY_DB_PATH), exist_ok=True)

# 依實際資料集大小動態計算 steps，避免「epoch 數從 10 改成 3，
# 但 eval/save steps 仍沿用舊的固定數字」導致整個 trial 期間
# 一次評估都沒觸發（pruner 因此完全拿不到資訊）
_train_size = len(train_ds["train"])
_steps_per_epoch = max(1, _train_size // SEARCH_BATCH_SIZE)
EVAL_STEPS_SEARCH = _steps_per_epoch          # 每個 epoch 評估一次 -> 3 次/trial，足夠 pruner 判斷
LOGGING_STEPS_SEARCH = max(1, _steps_per_epoch // 2)
WARMUP_STEPS_SEARCH = max(1, int(0.1 * _steps_per_epoch * SEARCH_EPOCHS))

print(f"訓練集大小: {_train_size}, 每個 epoch steps 數: {_steps_per_epoch}")

# ------------------------------------------------------------
# 6. Optuna Pruning Callback
#    把每次 evaluate 的 eval_loss 回報給 Optuna，
#    若這個 trial 明顯比其他 trial 差就提早中止，節省算力
# ------------------------------------------------------------
class OptunaPruningCallback(TrainerCallback):
    def __init__(self, trial: optuna.trial.Trial):
        self.trial = trial
        self._report_idx = 0

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return
        score = metrics.get("eval_loss")
        if score is None:
            return
        self.trial.report(score, step=self._report_idx)
        self._report_idx += 1
        if self.trial.should_prune():
            raise optuna.TrialPruned()

# ------------------------------------------------------------
# 7. Objective function
# ------------------------------------------------------------
def objective(trial: optuna.trial.Trial) -> float:
    global base_model

    # --- 7.1 取樣超參數 ---
    r = trial.suggest_categorical("r", [1, 2, 4, 8, 16, 32, 64])
    alpha_mult = trial.suggest_categorical("alpha_mult", [1.5, 2.0, 2.5, 3.0])
    lora_alpha = max(1, int(round(r * alpha_mult)))
    lora_dropout = trial.suggest_categorical("lora_dropout", [0.0, 0.1, 0.15, 0.2, 0.25])
    learning_rate = trial.suggest_float("learning_rate", 1e-6, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)

    print(f"\n=== Trial {trial.number} ===")
    print(f"r={r}, lora_alpha={lora_alpha}, lora_dropout={lora_dropout}, "
          f"lr={learning_rate:.2e}, weight_decay={weight_decay:.2e}")

    # 每個 trial 用相同 seed，確保結果差異只來自超參數本身
    set_seed(42)

    trial_output_dir = os.path.join(OUTPUT_ROOT, f"trial_{trial.number}")

    model = None
    trainer = None
    best_eval_loss = float("inf")

    try:
        # --- 7.2 包一層新的 LoRA adapter ---
        lora_config = LoraConfig(
            r=r,
            lora_alpha=lora_alpha,
            target_modules=TARGET_MODULES,
            lora_dropout=lora_dropout,
            bias="none",
        )
        model = get_peft_model(base_model, lora_config)
        model.train()

        training_args = Seq2SeqTrainingArguments(
            seed=42,
            data_seed=42,
            output_dir=trial_output_dir,
            per_device_train_batch_size=SEARCH_BATCH_SIZE,
            per_device_eval_batch_size=SEARCH_BATCH_SIZE,
            gradient_accumulation_steps=1,
            learning_rate=learning_rate,
            num_train_epochs=SEARCH_EPOCHS,
            gradient_checkpointing=False,
            fp16=True,
            weight_decay=weight_decay,
            lr_scheduler_type="linear",
            optim="adamw_torch",

            eval_strategy="steps",
            eval_steps=EVAL_STEPS_SEARCH,
            logging_steps=LOGGING_STEPS_SEARCH,
            warmup_steps=WARMUP_STEPS_SEARCH,

            # 搜尋階段不存檔，避免 50 個 trial 把 Drive 空間塞滿；
            # save_strategy="no" 時 load_best_model_at_end 必須是 False，
            # 否則 Trainer 會直接報錯
            save_strategy="no",
            load_best_model_at_end=False,

            predict_with_generate=False,  # 只看 loss，不做生成式評估，加快搜尋速度
            report_to=[],                  # 不寫 tensorboard log，避免產生大量小檔案
            disable_tqdm=True,
        )

        trainer = Seq2SeqTrainer(
            args=training_args,
            model=model,
            train_dataset=train_ds["train"],
            eval_dataset=test_ds["validation"],
            data_collator=data_collator,
            processing_class=processor,
            callbacks=[OptunaPruningCallback(trial)],
        )

        trainer.train()

        # 用整個 trial 過程中最低的 eval_loss 當作這組參數的分數，
        # 比只看最後一次評估更穩定（避免最後剛好抖動到比較差的點）
        eval_losses = [
            log["eval_loss"] for log in trainer.state.log_history
            if "eval_loss" in log
        ]
        if eval_losses:
            best_eval_loss = min(eval_losses)

    except optuna.TrialPruned:
        raise

    except RuntimeError as e:
        # 處理 CUDA Out Of Memory：不要讓單一 trial 把整個 50 次搜尋中斷，
        # 改成標記此 trial 為 pruned，讓 Optuna 繼續嘗試下一組參數
        if "out of memory" in str(e).lower():
            print(f"Trial {trial.number} 發生 CUDA OOM，標記為 pruned 並繼續下一組參數")
            torch.cuda.empty_cache()
            raise optuna.TrialPruned()
        raise

    finally:
        # --- 7.3 清理：把 base_model 還原成乾淨狀態，並釋放 GPU 記憶體 ---
        # LoRA 凍結 base weights，只要 unload() 移除 adapter，
        # base_model 就能在下一個 trial 重新使用，不必重新從硬碟讀檔
        if model is not None:
            try:
                base_model = model.unload()
            except Exception:
                pass
        if trainer is not None:
            del trainer
        del model
        gc.collect()
        torch.cuda.empty_cache()

    print(f"Trial {trial.number} best eval_loss = {best_eval_loss:.4f}")
    return best_eval_loss

# ------------------------------------------------------------
# 8. 建立（或續接）Optuna study
#    用 SQLite 檔案存在 Google Drive，即使 Colab 斷線重啟，
#    重新執行這個 cell 也能接續未完成的搜尋，不會從頭開始
# ------------------------------------------------------------
study = optuna.create_study(
    study_name="whisper_tiny_lora_search",
    storage=f"sqlite:///{STUDY_DB_PATH}",
    load_if_exists=True,
    direction="minimize",
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=1, interval_steps=1),
)

trials_done = len(study.trials)
trials_remaining = max(0, N_TRIALS - trials_done)
print(f"已完成 {trials_done} 個 trial，本次將再執行 {trials_remaining} 個 trial")

if trials_remaining > 0:
    try:
        study.optimize(
            objective,
            n_trials=trials_remaining,
            gc_after_trial=True,
            catch=(Exception,),  # 任何未預期例外都不會中斷整個搜尋，該 trial 標記 FAIL 後繼續下一個
        )
    except KeyboardInterrupt:
        print("偵測到手動中斷，已完成的 trial 結果已存在 SQLite 檔案中，可重新執行此 cell 繼續搜尋")

# ------------------------------------------------------------
# 9. 輸出搜尋結果
# ------------------------------------------------------------
completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
print(f"\n總共完成 {len(completed_trials)} 個有效 trial")

if completed_trials:
    print("最佳超參數組合：")
    print(json.dumps(study.best_params, indent=2, ensure_ascii=False))
    print(f"最佳 eval_loss: {study.best_value:.4f}")

    with open(BEST_PARAMS_PATH, "w", encoding="utf-8") as f:
        json.dump(
            {"best_params": study.best_params, "best_value": study.best_value},
            f, indent=2, ensure_ascii=False
        )
    print(f"已將最佳超參數存到 {BEST_PARAMS_PATH}")
else:
    print("目前沒有任何成功完成的 trial，請檢查上方錯誤訊息（可能是資料路徑或顯存不足）")

# ============================================================
# 10.（可選）用最佳超參數做完整訓練
#     此區塊預設不會自動執行，確認 study.best_params 沒問題後，
#     把 RUN_FINAL_TRAINING 改成 True 再執行
# ============================================================
RUN_FINAL_TRAINING = False

if RUN_FINAL_TRAINING and completed_trials:
    best = study.best_params
    final_r = best["r"]
    final_alpha = max(1, int(round(final_r * best["alpha_mult"])))
    final_dropout = best["lora_dropout"]
    final_lr = best["learning_rate"]
    final_wd = best["weight_decay"]

    set_seed(42)

    # base_model 在每個 trial 結束時都已經 unload() 還原成乾淨狀態，可直接重用
    final_lora_config = LoraConfig(
        r=final_r,
        lora_alpha=final_alpha,
        target_modules=TARGET_MODULES,
        lora_dropout=final_dropout,
        bias="none",
    )
    final_model = get_peft_model(base_model, final_lora_config)
    final_model.train()
    final_model.print_trainable_parameters()

    FINAL_EPOCHS = 10
    final_warmup_steps = max(1, int(0.1 * _steps_per_epoch * FINAL_EPOCHS))

    final_training_args = Seq2SeqTrainingArguments(
        seed=42,
        data_seed=42,
        output_dir="/content/drive/MyDrive/optuna/optuna_tcmWhisper",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=1,
        learning_rate=final_lr,
        num_train_epochs=FINAL_EPOCHS,
        gradient_checkpointing=False,
        fp16=True,
        weight_decay=final_wd,
        lr_scheduler_type="linear",
        optim="adamw_torch",

        eval_strategy="steps",
        save_strategy="steps",
        save_steps=_steps_per_epoch,
        eval_steps=_steps_per_epoch,
        logging_steps=_steps_per_epoch,
        warmup_steps=final_warmup_steps,

        predict_with_generate=False,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        report_to=["tensorboard"],
        push_to_hub=False,
    )

    final_trainer = Seq2SeqTrainer(
        args=final_training_args,
        model=final_model,
        train_dataset=train_ds["train"],
        eval_dataset=test_ds["validation"],
        data_collator=data_collator,
        processing_class=processor,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    final_model.generation_config.use_cache = False
    final_trainer.train()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

... (13 outputs left out)

[I 2026-06-20 10:02:01,990] Trial 1 finished with value: 0.7981446385383606 and parameters: {'r': 4, 'alpha_mult': 1.5, 'lora_dropout': 0.25, 'learning_rate': 0.0007886714129990489, 'weight_decay': 0.0017123375973163992}. Best is trial 1 with value: 0.7981446385383606.


Trial 1 best eval_loss = 0.7981

=== Trial 2 ===
r=4, lora_alpha=6, lora_dropout=0.2, lr=6.58e-04, weight_decay=3.80e-03
{'loss': '2.243', 'grad_norm': '2.009', 'learning_rate': '0.000618', 'epoch': '0.5'}
{'loss': '0.8632', 'grad_norm': '1.345', 'learning_rate': '0.0004967', 'epoch': '1'}
{'eval_loss': '0.8937', 'eval_runtime': '249.7', 'eval_samples_per_second': '6.733', 'eval_steps_per_second': '0.425', 'epoch': '1'}
{'loss': '0.5578', 'grad_norm': '0.9515', 'learning_rate': '0.0003754', 'epoch': '1.5'}
{'loss': '0.4876', 'grad_norm': '1.025', 'learning_rate': '0.0002541', 'epoch': '2'}
{'eval_loss': '0.8119', 'eval_runtime': '253.3', 'eval_samples_per_second': '6.637', 'eval_steps_per_second': '0.418', 'epoch': '2'}
{'loss': '0.4201', 'grad_norm': '0.9793', 'learning_rate': '0.0001328', 'epoch': '2.5'}
{'loss': '0.372', 'grad_norm': '1.104', 'learning_rate': '1.155e-05', 'epoch': '3'}
{'eval_loss': '0.8052', 'eval_runtime': '250.9', 'eval_samples_per_second': '6.699', 'eval_steps_p

[I 2026-06-20 10:20:13,682] Trial 2 finished with value: 0.8051886558532715 and parameters: {'r': 4, 'alpha_mult': 1.5, 'lora_dropout': 0.2, 'learning_rate': 0.0006584106160121611, 'weight_decay': 0.003795853142670641}. Best is trial 1 with value: 0.7981446385383606.


Trial 2 best eval_loss = 0.8052

=== Trial 3 ===
r=2, lora_alpha=4, lora_dropout=0.25, lr=2.07e-04, weight_decay=6.24e-06
{'loss': '2.82', 'grad_norm': '3.616', 'learning_rate': '0.0001965', 'epoch': '0.5'}
{'loss': '1.793', 'grad_norm': '1.75', 'learning_rate': '0.0001583', 'epoch': '1'}
